In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. LOAD DATA (edit path)
# -----------------------------
df = pd.read_stata("1.5kcolleges_student&collegedetails_v2_limitedvars.dta")


print("Loaded dataframe:", df.shape)

# -----------------------------
# 2. Normalizers
# -----------------------------
def norm_series(s):
    """Clean JC codes & preferences: lowercase, remove .0, handle empty."""
    s2 = (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.lower()
    )
    return s2.replace({
        "nan": np.nan, "": np.nan, "0": np.nan,
        "none": np.nan, "na": np.nan, "n/a": np.nan
    })

# Clean allotted codes
if "allotted_JC_code" not in df.columns:
    raise RuntimeError("Expected column 'allotted_JC_code' in df.")

allotted_clean = norm_series(df["allotted_JC_code"])

# Detect preference columns (includes 1–20)
pref_cols = [c for c in df.columns if "code_suffix_general_pref" in c.lower()]
pref_cols = pref_cols[:20]  # just first 20
if len(pref_cols) == 0:
    raise RuntimeError("No preference columns found.")

prefs_clean = df[pref_cols].apply(norm_series)

# Convert marks
marks = pd.to_numeric(df["marks"], errors="coerce")


Loaded dataframe: (39435, 278)


In [4]:
# -------------------------
# BUILD ORDERED PAIR-BINS (A > B) WHERE:
# - at least 5 students listed BOTH A and B AND ranked A above B
# - among those students, they were allotted to either A or B
# -------------------------
import pandas as pd
import numpy as np
from collections import defaultdict
import math
import time

t0 = time.time()

# ---- REQUIREMENTS: df must be loaded ----
if "df" not in globals():
    raise RuntimeError("Dataframe `df` not found in global namespace. Load your data first.")

# ---- Normalized allotted codes and preference columns (reuse if present) ----
def norm_series(s):
    s2 = (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.lower()
    )
    return s2.replace({
        "nan": np.nan, "": np.nan, "0": np.nan,
        "none": np.nan, "na": np.nan, "n/a": np.nan
    })

# allotted_clean (reuse if present)
if "allotted_clean" in globals():
    allotted = allotted_clean
else:
    if "allotted_JC_code" not in df.columns:
        raise RuntimeError("Expected column 'allotted_JC_code' in df.")
    allotted = norm_series(df["allotted_JC_code"])

# detect preference columns
pref_cols = [c for c in df.columns if "code_suffix_general_pref" in c.lower()]
pref_cols = pref_cols[:20]  # limit to first 20 preferences
if len(pref_cols) == 0:
    raise RuntimeError("No preference columns found (expected 'code_suffix_general_pref' pattern).")

# prefs_clean (reuse if present)
if "prefs_clean" in globals():
    prefs = prefs_clean
else:
    prefs = df[pref_cols].apply(norm_series)

# ---- Build code -> {idx: rank} map (fast single pass over pref columns) ----
code_to_rank = {}       # code -> dict(idx -> rank)
code_to_applicants = defaultdict(set)  # code -> set of indices who listed it

for rank, col in enumerate(pref_cols, start=1):
    ser = prefs[col]
    # dropna for speed
    nonnull = ser[ser.notna()]
    for idx, code in nonnull.items():
        # initialize dict lazily
        if code not in code_to_rank:
            code_to_rank[code] = {}
        # only set if not already set by an earlier (higher priority) pref column
        # but since we iterate in increasing rank, we can simply set (first occurrence is lowest rank number)
        if idx not in code_to_rank[code]:
            code_to_rank[code][idx] = rank
            code_to_applicants[code].add(idx)

# ---- Candidate JC codes: union of codes appearing in prefs and allotted ----
codes_in_prefs = set(code_to_applicants.keys())
codes_in_allotted = set(alotted for alotted in allotted.dropna().unique())
candidate_codes = sorted(list(codes_in_prefs.union(codes_in_allotted)))

print(f"Total unique JC codes (prefs ∪ allotted): {len(candidate_codes)}")

# ---- Prepare quick access to allotted values as a numpy array for boolean indexing ----
allotted_series = allotted  # pandas Series aligned with df.index

# ---- Main loop: iterate ordered pairs (A,B) and collect qualifying pairs ----
min_required = 5
pair_bins = {}        # pair_id -> {jc1, jc2, n_matched, idxs_list, n_jc1, n_jc2}
summary_rows = []

n_codes = len(candidate_codes)
count_pairs = 0
# We will iterate ordered pairs A != B
for i, A in enumerate(candidate_codes):
    # small micro-optimization: skip codes with very few applicants overall
    if len(code_to_applicants.get(A, set())) < 1:
        continue
    for B in candidate_codes:
        if B == A:
            continue
        # fast intersection of applicant sets
        setA = code_to_applicants.get(A, set())
        setB = code_to_applicants.get(B, set())
        if not setA or not setB:
            continue
        common = setA.intersection(setB)
        if len(common) < min_required:
            continue  # not enough who even listed both

        # Now filter those who ranked A higher than B
        matched_idxs = []
        for idx in common:
            rankA = code_to_rank.get(A, {}).get(idx, math.nan)
            rankB = code_to_rank.get(B, {}).get(idx, math.nan)
            if math.isnan(rankA) or math.isnan(rankB):
                continue
            # smaller rank -> higher preference
            if rankA < rankB:
                # only include if allotted to A or B
                final = allotted_series.loc[idx]
                if pd.isna(final):
                    continue
                if final == A or final == B:
                    matched_idxs.append(idx)

        if len(matched_idxs) >= min_required:
            # create pair id (directional). Use .1 suffix to indicate A>B direction
            pair_id = f"{A}x{B}.1"
            n_total = len(matched_idxs)
            # counts by allotment destination
            matched_idxs_series = pd.Index(matched_idxs)
            n_to_A = int((allotted_series.loc[matched_idxs_series] == A).sum())
            n_to_B = int((allotted_series.loc[matched_idxs_series] == B).sum())

            pair_bins[pair_id] = {
                "jc1": A,
                "jc2": B,
                "n_matched": n_total,
                "idxs": matched_idxs_series,   # store Index object (in-memory only)
                "n_to_jc1": n_to_A,
                "n_to_jc2": n_to_B
            }

            summary_rows.append({
                "pair_id": pair_id,
                "jc1": A,
                "jc2": B,
                "n_matched": n_total,
                "n_to_jc1": n_to_A,
                "n_to_jc2": n_to_B
            })
            count_pairs += 1

# Build summary DataFrame
pairs_summary = pd.DataFrame(summary_rows)
pairs_summary = pairs_summary.sort_values("n_matched", ascending=False).reset_index(drop=True)

t1 = time.time()
print(f"Done. Found {count_pairs} ordered pairs meeting the threshold (≥ {min_required}).")
print(f"Elapsed time: {t1 - t0:.1f} seconds")

# show top results
if not pairs_summary.empty:
    display(pairs_summary.head(20))

# pair_bins contains the detailed match indices for each pair_id (in-memory)
# pairs_summary is a tidy DataFrame summarizing the pairs

# Keep objects in namespace: pair_bins, pairs_summary


Total unique JC codes (prefs ∪ allotted): 234
Done. Found 4762 ordered pairs meeting the threshold (≥ 5).
Elapsed time: 139.7 seconds


,pair_id,jc1,jc2,n_matched,n_to_jc1,n_to_jc2
0,501502_bpcx501903_mec.1,501502_bpc,501903_mec,78,65,13
1,501502_bpcx501904_cec.1,501502_bpc,501904_cec,74,64,10
2,501502_bpcx501802_bpc.1,501502_bpc,501802_bpc,69,61,8
3,507402_bpcx507702_bpc.1,507402_bpc,507702_bpc,66,37,29
4,507502_bpcx507702_bpc.1,507502_bpc,507702_bpc,65,36,29
5,501502_bpcx502403_mec.1,501502_bpc,502403_mec,62,62,0
6,508102_bpcx507702_bpc.1,508102_bpc,507702_bpc,62,34,28
7,501502_bpcx504903_mec.1,501502_bpc,504903_mec,62,62,0
8,501502_bpcx502102_bpc.1,501502_bpc,502102_bpc,61,55,6
9,501502_bpcx505203_mec.1,501502_bpc,505203_mec,61,61,0


In [6]:

import math

# -----------------------------
# REQUIREMENTS CHECK
# -----------------------------
for v in ["df", "pair_bins", "allotted_clean", "marks"]:
    if v not in globals():
        raise RuntimeError(f"Required object '{v}' not found")

# -----------------------------
# PARAMETERS
# -----------------------------
BIN_WIDTH = 10

# -----------------------------
# STORAGE (LONG FORM)
# -----------------------------
long_rows = []

# -----------------------------
# MAIN LOOP
# -----------------------------
for pair_id, info in pair_bins.items():
    jc1 = info["jc1"]
    jc2 = info["jc2"]
    idxs = info["idxs"]          # pandas Index of student indices

    # Extract marks and allotments for this pair
    pair_marks = marks.loc[idxs].dropna()
    pair_allotted = allotted_clean.loc[pair_marks.index]

    if pair_marks.empty:
        continue

    # Compute bins: left-inclusive, right-exclusive
    # bin_lower = floor(mark / 5) * 5
    bin_lower = (pair_marks // BIN_WIDTH) * BIN_WIDTH
    bin_upper = bin_lower + (BIN_WIDTH - 1)

    # Build temp DataFrame
    temp = pd.DataFrame({
        "candidate_id_68k": pair_marks.index,
        "marks": pair_marks.values,
        "bin_lo": bin_lower.values.astype(int),
        "bin_hi": bin_upper.values.astype(int),
        "allotted": pair_allotted.values
    })

    # Group by bins
    for (lo, hi), g in temp.groupby(["bin_lo", "bin_hi"]):
        # Check presence of BOTH colleges
        has_jc1 = (g["allotted"] == jc1).any()
        has_jc2 = (g["allotted"] == jc2).any()

        if not (has_jc1 and has_jc2):
            continue

        # Construct bin_id
        bin_id = f"{pair_id}__marks_{lo}_{hi}"

        # Store long-form rows
        for sid in g["candidate_id_68k"]:
            long_rows.append({
                "pair_id": pair_id,
                "bin_id": bin_id,
                "student_id": df.loc[sid, "candidate_id_68k"]
            })

# -----------------------------
# BUILD FINAL LONG DATAFRAME
# -----------------------------
pair_bin_long = pd.DataFrame(long_rows)

# -----------------------------
# REPORTING
# -----------------------------
n_bins = pair_bin_long["bin_id"].nunique()
n_pairs = len(pair_bins)

print("Number of JC pairs in dataset:", n_pairs)
print("Number of qualifying (pair × marks-bin) bins:", n_bins)

# Optional: preview
display(pair_bin_long.head(10))


Number of JC pairs in dataset: 4762
Number of qualifying (pair × marks-bin) bins: 5276


,pair_id,bin_id,student_id
0,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010001167
1,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010059735
2,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010030237
3,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010048560
4,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010039326
5,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010055098
6,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010012955
7,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010013135
8,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010013275
9,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010013549


In [7]:
pair_bin_long.to_csv("pair_bin_long_10.csv", index=False)


In [8]:
ID_COL = "candidate_id_68k"
BIN_WIDTH = 10
MIN_REQUIRED = 5


In [9]:
def norm_series(s):
    s = (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.lower()
    )
    return s.replace({
        "nan": np.nan, "": np.nan, "0": np.nan,
        "none": np.nan, "na": np.nan, "n/a": np.nan
    })

# core columns
marks = pd.to_numeric(df["marks"], errors="coerce")
allotted = norm_series(df["allotted_JC_code"])

# preferences
pref_cols = [c for c in df.columns if "code_suffix_general_pref" in c.lower()][:20]
prefs = df[pref_cols].apply(norm_series)


In [10]:
# student_idx → {jc_code: rank}
pref_rank = {}

# jc_code → set(student_idx)
code_to_students = {}

for rank, col in enumerate(pref_cols, start=1):
    for idx, code in prefs[col].dropna().items():
        pref_rank.setdefault(idx, {})
        code_to_students.setdefault(code, set())
        if code not in pref_rank[idx]:
            pref_rank[idx][code] = rank
            code_to_students[code].add(idx)

all_codes = sorted(code_to_students)


In [11]:
# base student table
student_wide = (
    df[[ID_COL, "marks", "gtot", "allotted_JC_code"]]
    .drop_duplicates(subset=[ID_COL])
    .set_index(ID_COL)
)

# enforce valid numeric IDs
student_wide = student_wide[
    student_wide.index.astype(str).str.match(r"^\d+$")
]

# container for bin memberships
student_bins = {sid: [] for sid in student_wide.index}


In [12]:
for A in all_codes:
    for B in all_codes:
        if A == B:
            continue

        # students who listed both
        common = code_to_students[A] & code_to_students[B]
        if len(common) < MIN_REQUIRED:
            continue

        # students with A > B preference AND allotted to A or B
        aligned = []
        for idx in common:
            if pref_rank[idx][A] < pref_rank[idx][B]:
                if allotted.loc[idx] in (A, B):
                    aligned.append(idx)

        if len(aligned) < MIN_REQUIRED:
            continue

        # bin these students by marks
        aligned_marks = marks.loc[aligned].dropna()
        aligned_allotted = allotted.loc[aligned_marks.index]

        bin_lo = (aligned_marks // BIN_WIDTH) * BIN_WIDTH
        bin_hi = bin_lo + (BIN_WIDTH - 1)

        temp = pd.DataFrame({
            "idx": aligned_marks.index,
            "lo": bin_lo.astype(int),
            "hi": bin_hi.astype(int),
            "allotted": aligned_allotted.values
        })

        pair_id = f"{A}x{B}.1"

        for (lo, hi), g in temp.groupby(["lo", "hi"]):
            if not ((g["allotted"] == A).any() and (g["allotted"] == B).any()):
                continue

            bin_id = f"{pair_id}__marks_{lo}_{hi}"

            for idx in g["idx"]:
                sid = df.loc[idx, ID_COL]
                if sid in student_bins:
                    student_bins[sid].append(bin_id)


In [13]:
max_bins = max(len(v) for v in student_bins.values())

bin_cols = [f"pair_bin_{i+1}" for i in range(max_bins)]

bin_wide = pd.DataFrame(
    [
        student_bins[sid] + [np.nan] * (max_bins - len(student_bins[sid]))
        for sid in student_wide.index
    ],
    index=student_wide.index,
    columns=bin_cols
)


In [14]:
student_wide_direct = pd.concat([student_wide, bin_wide], axis=1)

print("Direct wide dataset shape:", student_wide_direct.shape)
print("Max bins per student:", max_bins)

display(student_wide_direct.head())


Direct wide dataset shape: (39435, 22)
Max bins per student: 19


,marks,gtot,allotted_JC_code,pair_bin_1,pair_bin_2,pair_bin_3,pair_bin_4,pair_bin_5,pair_bin_6,pair_bin_7,...,pair_bin_10,pair_bin_11,pair_bin_12,pair_bin_13,pair_bin_14,pair_bin_15,pair_bin_16,pair_bin_17,pair_bin_18,pair_bin_19
candidate_id_68k,,,,,,,,,,,,,,,,,,,,,
2010000012,65,784.0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010000013,(null),650.0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010000014,22.5,734.0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010000018,50.25,885.0,508402_BPC,508402_bpcx508801_mpc.1__marks_50_59,508402_bpcx509301_mpc.1__marks_50_59,508402_bpcx509302_bpc.1__marks_50_59,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010000019,13.75,385.0,511901_MPC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
student_wide_direct.to_csv("student_wide_direct_10.csv")


In [16]:
student_wide_direct = student_wide_direct.reset_index()

# -----------------------------
# FORCE CONSISTENT ID TYPE
# -----------------------------
student_wide_direct[ID_COL] = student_wide_direct[ID_COL].astype(str)
df[ID_COL] = df[ID_COL].astype(str)

# -----------------------------
# VARIABLES TO ADD (unchanged)
# -----------------------------
vars_to_add = [
    "englishmarks_12", "marks12th", "dob_year", "age", "female",
    "social_category", "region", "minority", "religion",
    "english_medium_10", "english_medium_9",
    "school_type_9", "school_type_10",
    "type", "appliedforJC", "result",
    "address_pincode", "address_mandalname", "address_district",
    "coll_code_went", "got_into_JC", "TSJC",
    "intermediatecollegecode", "sixdigitcode",
    "coll_code", "institution",
    "is_income", "maxtot", "th_mrk", "pr_mrk", "tot_mrk"
]

vars_to_add = [v for v in vars_to_add if v in df.columns]

# -----------------------------
# MERGE
# -----------------------------
final_rudimentary_grouping_10 = (
    student_wide_direct
    .merge(
        df[[ID_COL] + vars_to_add],
        on=ID_COL,
        how="left"
    )
)

# -----------------------------
# FINAL CHECK
# -----------------------------
assert final_rudimentary_grouping_10[ID_COL].is_unique, "Primary key not unique"

print("Final dataset shape:", final_rudimentary_grouping_10.shape)
print("Added variables:", len(vars_to_add))

display(final_rudimentary_grouping_10.head())


Final dataset shape: (39435, 54)
Added variables: 31


,candidate_id_68k,marks,gtot,allotted_JC_code,pair_bin_1,pair_bin_2,pair_bin_3,pair_bin_4,pair_bin_5,pair_bin_6,...,TSJC,intermediatecollegecode,sixdigitcode,coll_code,institution,is_income,maxtot,th_mrk,pr_mrk,tot_mrk
0,2010000012,65,784.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,,54018.0,,Y,1000,070,0,070
1,2010000013,(null),650.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,,56234.0,,Y,1000,060,0,060
2,2010000014,22.5,734.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,56024.0,505203,56024.0,,Y,1000,,,
3,2010000018,50.25,885.0,508402_BPC,508402_bpcx508801_mpc.1__marks_50_59,508402_bpcx509301_mpc.1__marks_50_59,508402_bpcx509302_bpc.1__marks_50_59,NaN,NaN,NaN,...,1.0,50018.0,508401,50018.0,tswrs/jc(g)\nkondamallepally,Y,1000,096,0,096
4,2010000019,13.75,385.0,511901_MPC,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,42008.0,511901,42008.0,tswrs/jc(b) jangaon,Y,1000,061,0,061


In [18]:
final_rudimentary_grouping_10.to_csv("final_rudimentary_grouping_10.csv", index=False)


In [19]:
import pandas as pd

# -----------------------------
# LOAD DATASETS
# -----------------------------
pair_bin_long = pd.read_csv("pair_bin_long_10.csv")
student_wide = pd.read_csv("final_rudimentary_grouping_10.csv")

# Ensure IDs are strings
pair_bin_long["student_id"] = pair_bin_long["student_id"].astype(str)
student_wide["candidate_id_68k"] = student_wide["candidate_id_68k"].astype(str)

# -----------------------------
# CONSTRUCT REGRESSION DATA
# -----------------------------
reg_df = (
    pair_bin_long
    .merge(
        student_wide[["candidate_id_68k", "gtot", "allotted_JC_code", "englishmarks_12"]],
        left_on="student_id",
        right_on="candidate_id_68k",
        how="left"
    )
)

# Rename for clarity
reg_df = reg_df.rename(columns={
    "bin_id": "pair_bin",
    "allotted_JC_code": "college_attended"
})

print("Regression-ready dataframe shape:", reg_df.shape)
display(reg_df.head())


Regression-ready dataframe shape: (31980, 7)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_31148\1385584037.py:7: DtypeWarning: Columns (51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  student_wide = pd.read_csv("final_rudimentary_grouping_10.csv")


,pair_id,pair_bin,student_id,candidate_id_68k,gtot,college_attended,englishmarks_12
0,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010001167,2010001167,742.0,501601_MPC,94.0
1,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010059735,2010059735,838.0,501502_BPC,71.0
2,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010030237,2010030237,770.0,501601_MPC,86.0
3,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010048560,2010048560,938.0,501502_BPC,90.0
4,501502_bpcx501601_mpc.1,501502_bpcx501601_mpc.1__marks_20_29,2010039326,2010039326,884.0,501502_BPC,78.0


In [20]:
reg_df.groupby("pair_bin")["college_attended"].nunique().value_counts()


college_attended
2    5276
Name: count, dtype: int64

In [21]:
# 1. Drop missing outcome / college
reg_df = reg_df.dropna(subset=["gtot", "college_attended", "pair_bin", "englishmarks_12"])

# 2. Keep only bins with 2 colleges represented
bin_counts = (
    reg_df
    .groupby("pair_bin")["college_attended"]
    .nunique()
)

valid_bins = bin_counts[bin_counts >= 2].index

reg_df = reg_df[reg_df["pair_bin"].isin(valid_bins)]

print("After filtering, regression df shape:", reg_df.shape)


After filtering, regression df shape: (31980, 7)


In [22]:
reg_df[reg_df["pair_bin"].isin(valid_bins)] \
      .groupby("pair_bin")["college_attended"].nunique().value_counts()

college_attended
2    5276
Name: count, dtype: int64

In [23]:
before = pair_bin_long["student_id"].nunique()
after = reg_df["student_id"].nunique()

print("Unique students before:", before)
print("Unique students after:", after)


Unique students before: 5435
Unique students after: 5435


In [24]:
import statsmodels.formula.api as smf

# Ensure categorical types
reg_df["pair_bin"] = reg_df["pair_bin"].astype("category")
reg_df["college_attended"] = reg_df["college_attended"].astype("category")

print("Number of pair bins:", reg_df["pair_bin"].nunique())
print("Number of colleges:", reg_df["college_attended"].nunique())


Number of pair bins: 5276
Number of colleges: 193


In [25]:
reg_df.to_csv("reg_df_10.csv")

In [26]:
import statsmodels.api as sm
import pandas as pd

# -----------------------------
# ENSURE STRING TYPES
# -----------------------------
reg_df["pair_bin"] = reg_df["pair_bin"].astype(str)
reg_df["college_attended"] = ( reg_df["college_attended"]
    .astype(str)
    .str.strip()
    .str.lower()
                             )

# -----------------------------
# WITHIN-BIN DEMEANING
# -----------------------------
reg_df["gtot_dm"] = (
    reg_df["gtot"]
    - reg_df.groupby("pair_bin")["gtot"].transform("mean")
)

# -----------------------------
# COLLEGE DUMMIES (NO INTERCEPT)
# -----------------------------
college_dummies = pd.get_dummies(
    reg_df["college_attended"],
    drop_first=True   # baseline dropped automatically
)

X = college_dummies
y = reg_df["gtot_dm"]

# -----------------------------
# CLUSTERED OLS
# -----------------------------
model = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["pair_bin"]}
)

print(model.summary())


                                 OLS Regression Results                                
Dep. Variable:                gtot_dm   R-squared (uncentered):                   0.064
Model:                            OLS   Adj. R-squared (uncentered):              0.059
Method:                 Least Squares   F-statistic:                              12.24
Date:                Tue, 23 Dec 2025   Prob (F-statistic):                   1.25e-293
Time:                        12:10:22   Log-Likelihood:                     -1.9463e+05
No. Observations:               31980   AIC:                                  3.897e+05
Df Residuals:                   31788   BIC:                                  3.913e+05
Df Model:                         192                                                  
Covariance Type:              cluster                                                  
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------

In [27]:
import statsmodels.api as sm
import pandas as pd

# -----------------------------
# ENSURE STRING TYPES
# -----------------------------
reg_df["pair_bin"] = reg_df["pair_bin"].astype(str)
reg_df["college_attended"] = ( reg_df["college_attended"]
    .astype(str)
    .str.strip()
    .str.lower()
                             )

# -----------------------------
# WITHIN-BIN DEMEANING
# -----------------------------
reg_df["englishmarks_12_dm"] = (
    reg_df["englishmarks_12"]
    - reg_df.groupby("pair_bin")["englishmarks_12"].transform("mean")
)

# -----------------------------
# COLLEGE DUMMIES (NO INTERCEPT)
# -----------------------------
college_dummies = pd.get_dummies(
    reg_df["college_attended"],
    drop_first=True   # baseline dropped automatically
)

X = college_dummies
y = reg_df["englishmarks_12_dm"]

# -----------------------------
# CLUSTERED OLS
# -----------------------------
model = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["pair_bin"]}
)

print(model.summary())


                                 OLS Regression Results                                
Dep. Variable:     englishmarks_12_dm   R-squared (uncentered):                   0.032
Model:                            OLS   Adj. R-squared (uncentered):              0.026
Method:                 Least Squares   F-statistic:                              6.844
Date:                Tue, 23 Dec 2025   Prob (F-statistic):                   3.21e-144
Time:                        12:11:08   Log-Likelihood:                     -1.2127e+05
No. Observations:               31980   AIC:                                  2.429e+05
Df Residuals:                   31788   BIC:                                  2.445e+05
Df Model:                         192                                                  
Covariance Type:              cluster                                                  
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------